# The REG118_PREV_FILED_APPLN table

Welcome to a comprehensive exploration of one of the key tables in the PATSTAT database: the `REG118_PREV_FILED_APPLN` table. This table plays a pivotal role in tracking previously filed patent applications, providing insights into the continuity and progression of patent families across jurisdictions and time.

The table includes essential attributes such as `ID` (a unique numerical identifier for an application), `APPLN_AUTH` (the authority where the application was filed), `APPLN_NR` (the application number as issued by the filing authority), and `APPLN_DATE` (the date of filing). It also features publication-related information like `BULLETIN_YEAR` and `BULLETIN_NR`, indicating the year and calendar week of the European Patent Bulletin in which the relevant actions were published.

By linking `REG118_PREV_FILED_APPLN` with other tables, such as `REG101_APPLN` (which contains information on current applications) or `REG201_PROC_STEP` (which details procedural steps), we can gain a deeper understanding of how applications evolve and are connected within patent families. This linkage helps map the lifecycle of patent applications, from filing through to publication and beyond.

The `APPLN_AUTH` and `APPLN_DATE` fields enable geographic and temporal analysis of patent filings. For instance, researchers can identify trends in filing behaviours across regions or examine the impact of filing dates on the grant process. The `BULLETIN_YEAR` and `BULLETIN_NR` attributes provide a direct connection to European Patent Bulletin publications, enabling users to pinpoint the timing and context of key events.

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG118_PREV_FILED_APPLN, REG101_APPLN
from sqlalchemy import select, func, case, select, and_

patstat = PatstatClient(env='PROD')

db = patstat.orm()

In [2]:
q = db.query(
    REG118_PREV_FILED_APPLN.id,
    REG118_PREV_FILED_APPLN.bulletin_year,
    REG118_PREV_FILED_APPLN.bulletin_nr,
    REG118_PREV_FILED_APPLN.appln_auth,
    REG118_PREV_FILED_APPLN.appln_nr,
    REG118_PREV_FILED_APPLN.appln_date,
    REG101_APPLN.appln_filing_date
).join(
    REG101_APPLN, REG118_PREV_FILED_APPLN.id == REG101_APPLN.id  # Join on ID
)

res = patstat.df(q)
res

,id,bulletin_year,bulletin_nr,appln_auth,appln_nr,appln_date,appln_filing_date
0,17201391,2018,19,WO,2012EP54449,2012-03-14,2012-03-14
1,10193424,2011,20,EP,20060123268,2006-10-31,2006-10-31
2,19163620,2019,51,WO,2016US16944,2016-02-08,2016-02-08
3,12152120,2012,21,WO,2004CA00543,2004-04-08,2004-04-08
4,8000249,2008,32,DE,20071004779,2007-01-31,2008-01-09
...,...,...,...,...,...,...,...
6900,11164200,2011,42,EP,20090780542,2009-07-14,2009-07-14
6901,18197475,2019,35,JP,20130051193,2013-01-22,2013-01-22
6902,18184991,2019,20,WO,2013US57350,2013-08-29,2013-08-29
6903,22202578,2023,19,US,202163263625P,2021-11-05,2022-10-19


## Key Fields in the ``REG118_PREV_FILED_APPLN`` table

### ID (primary key)

The ID field serves as a technical identifier that uniquely connects patent applications across various tables. The ID is essential for linking application data with other related information.
ID values are derived systematically, ensuring that a specific application maintains the same ID in all PATSTAT editions. For European Patent (EP) applications, the ID is derived from the XML attribute id by removing the prefix "EP," suffix "P," and any leading zeros.

### BULLETIN_YEAR

In the PATSTAT database, the `BULLETIN_YEAR` field captures the year when an action or event related to a patent application was published in the European Patent Bulletin.

The `BULLETIN_YEAR` is a 4-digit numeric field (formatted as YYYY), with a default value of 0 to indicate cases where no European Patent Bulletin publication is known. For entries where publication in the European Patent Bulletin is confirmed, `BULLETIN_YEAR` reflects the corresponding year of publication. It is used in conjunction with `BULLETIN_NR`, which specifies the European Patent Bulletin issue number.

### BULLETIN_NR

The ``BULLETIN_NR`` attribute represents the issue number of the European Patent Bulletin in which a specific action has been published. This number indicates the calendar week during which the European Patent Bulletin was released. It serves as a reference for identifying the exact edition of the European Patent Bulletin where actions such as patent grants, publications, or other significant events are announced.

If the action was not published in the European Patent Bulletin or if the information is unknown, the default value of 0 is used for the ``BULLETIN_NR``, which corresponds to the absence of a known European Patent Bulletin number. This value is only used when the associated ``BULLETIN_YEAR`` is also set to 0.

In [3]:
q = db.query(
    REG118_PREV_FILED_APPLN.id,
    REG118_PREV_FILED_APPLN.bulletin_year,
    REG118_PREV_FILED_APPLN.bulletin_nr,
    REG118_PREV_FILED_APPLN.appln_auth,
    REG118_PREV_FILED_APPLN.appln_nr,
    REG118_PREV_FILED_APPLN.appln_date,
).filter(
    REG118_PREV_FILED_APPLN.bulletin_year == 2014  
).order_by(
    REG118_PREV_FILED_APPLN.bulletin_nr  
)

res = patstat.df(q)
res

,id,bulletin_year,bulletin_nr,appln_auth,appln_nr,appln_date
0,13172894,2014,1,US,201213538577,2012-06-29
1,13158706,2014,1,WO,2009IB51047,2009-03-13
2,14176023,2014,1,WO,2009EP64726,2009-11-06
3,13162439,2014,1,WO,2007CN02967,2007-10-16
4,13172863,2014,1,US,201213535438,2012-06-28
...,...,...,...,...,...,...
317,13177105,2014,51,EP,20130171748,2013-06-12
318,14173503,2014,51,EP,20130155293,2013-02-14
319,14181462,2014,52,EP,20120165218,2008-05-19
320,14169054,2014,52,EP,20070766873,2007-07-05


### APPLN_AUTH

The `APPLN_AUTH` attribute denotes the authority or office where a patent application has been filed.
For other patent applications, the `APPLN_AUTH` field may represent different authorities, depending on the jurisdiction where the application was submitted. The value is typically a two-character code, assigned according to the WIPO ST.3 standard, which defines country and regional office codes.

This attribute is essential for identifying the filing authority behind each patent application, providing insights into the geographical and institutional origins of patent filings. The default value is not applicable, as the `APPLN_AUTH` field is always populated with the respective authority code, ensuring clarity on where an application has been officially lodged.

In [4]:
q = db.query(
    REG118_PREV_FILED_APPLN.appln_auth,
    func.count(REG118_PREV_FILED_APPLN.appln_nr).label('application_count')
).group_by(
    REG118_PREV_FILED_APPLN.appln_auth
).order_by(
    func.count(REG118_PREV_FILED_APPLN.appln_nr).desc()  # Order by count in descending order
)


res = patstat.df(q)
res


,appln_auth,application_count
0,WO,3716
1,EP,2445
2,DE,263
3,US,197
4,PL,77
5,FR,29
6,CN,27
7,ES,23
8,GB,15
9,JP,15


### APPLN_NR

The `APPLN_NR` attribute represents the application number assigned by the application authority. This unique identifier is issued by the relevant patent office and is used to track and reference a specific patent application.

There is no default value for this attribute, as each application is assigned a unique number upon filing, and this number must be provided for each record.

The domain for this attribute consists of a string of 8 digits. Leading zeros are significant.

### APPLN_DATE

The `APPLN_DATE` attribute represents the date when a patent application was filed. In the `REG118_PREV_FILED_APPLN` table, this field captures the exact filing date for each application, providing temporal information for tracking the progress of patent filings over time.

The default value for this attribute is set to 9999-12-31, which serves as a placeholder to indicate an unknown or unspecified filing date.